# PPO experiment with a detector-gated two-head reward correction

This notebook tests the actual research hypothesis: **does predicted proxy–judge disagreement make PPO harder to reward-hack?**

The design avoids the extra-training confound:

1. Freeze proxy/judge calibration.
2. Train a stage-1 PPO policy with normalized proxy reward.
3. Collect 20k responses from that policy and train the selected two-head model (`alpha=3`, `lambda=1`).
4. Select the detector threshold on validation only (target recall >= 70%).
5. Continue stage-1 PPO twice from the identical checkpoint and dataset: proxy-only control vs corrected-reward treatment.
6. Compare both policies on identical held-out prompts using proxy, judge, reward-gap, detector, paired-bootstrap, and optional Prometheus metrics.

Correction used during PPO:

`r_corrected = r_proxy - I[p(high gap) >= tau] * clamp(d_hat, 0, cap) * proxy_std`

The non-negative clamp prevents a negative gap prediction from becoming a bonus; the training-derived cap limits exploitable extrapolation.

In [ ]:
from dataclasses import replace
import gc
import json
import logging
from pathlib import Path
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score, f1_score, mean_absolute_error, mean_squared_error,
    precision_score, recall_score, roc_auc_score,
)
import torch
from transformers import AutoTokenizer, set_seed

import functions
from Datasets.dataset_gap_finder import DatasetGapFinder
from Datasets.dataset_request import RequestDataset
from Models.lora import LoRASettings
from Models.model_policy import PolicyModel
from Models.model_reward import RewardModel
from Models.model_two_head_gap_finder import TwoHeadGapFinder
from Models.reward_adjustment import TwoHeadGapFinderCorrection
from Trainers.trainer_two_head_gap_finder import (
    TwoHeadGapFinderTrainer, TwoHeadGapFinderTrainingConfig, compute_two_head_report,
)
from functions import (
    ConfigTrainClassifier, DatasetSpec, EvaluateConfig, EvaluatorSpec,
    PolicySpec, RewardSpec, TrainingPPOConfig,
)

logging.getLogger().setLevel(logging.INFO)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)

## 1. Frozen experimental protocol

All dataset ranges are disjoint. Defaults are intended for one H200. For a smoke test, reduce all range sizes consistently; serious conclusions require the full run and preferably multiple random seeds.

In [ ]:
DATASET_NAME = "Anthropic/hh-rlhf"
MODEL_NAME = "Qwen/Qwen3-0.6B"
RANDOM_STATE = 42
EVALUATION_SEED = 2026

CALIBRATION_RANGE = (0, 5_000)
STAGE1_PPO_RANGE = (5_000, 10_000)
GAP_DATA_RANGE = (10_000, 30_000)
STAGE2_PPO_RANGE = (30_000, 35_000)
FINAL_EVAL_RANGE = (35_000, 36_000)
assert all(a[1] <= b[0] for a, b in zip(
    [CALIBRATION_RANGE, STAGE1_PPO_RANGE, GAP_DATA_RANGE, STAGE2_PPO_RANGE],
    [STAGE1_PPO_RANGE, GAP_DATA_RANGE, STAGE2_PPO_RANGE, FINAL_EVAL_RANGE],
))

HIGH_GAP_WEIGHT = 3.0
DETECTOR_LOSS_WEIGHT = 1.0
MIN_VALIDATION_RECALL = 0.70
THRESHOLDS = np.round(np.arange(0.01, 1.00, 0.01), 2)
CORRECTION_CAP_QUANTILE = 0.995
RUN_PROMETHEUS_EVALUATION = True

OUTPUT_ROOT = Path("outputs/two_head_gap_finder_ppo")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CALIBRATION_PATH = OUTPUT_ROOT / "calibration.json"
STAGE1_PATH = OUTPUT_ROOT / "stage1_proxy_ppo" / "final"
GAP_DATA_PATH = OUTPUT_ROOT / "stage1_gap_data_20k.json"
TWO_HEAD_RUN = OUTPUT_ROOT / "two_head_alpha=3_lambda=1"
TWO_HEAD_PATH = TWO_HEAD_RUN / "final"
CONTROL_PATH = OUTPUT_ROOT / "stage2_proxy_control" / "final"
TREATMENT_PATH = OUTPUT_ROOT / "stage2_two_head_corrected" / "final"

policy_spec = PolicySpec(model_name=MODEL_NAME)
proxy_spec = RewardSpec(
    class_name="RewardModel", model_name="Skywork/Skywork-Reward-V2-Qwen3-0.6B",
    mode_name="proxy",
)
judge_spec = RewardSpec(
    class_name="RewardModel", model_name="Skywork/Skywork-Reward-V2-Qwen3-4B",
    mode_name="judge",
)

def policy_checkpoint_ready(path):
    path = Path(path)
    return path.is_dir() and (
        (path / "adapter_config.json").is_file() or (path / "config.json").is_file()
    )

def release(*owners):
    for owner in owners:
        functions.offload_to_cpu(owner)
    gc.collect()
    functions.empty_cuda_cache()

## 2. Freeze reference calibration

The same proxy/judge mean and standard deviation are used for every later comparison. Never recalculate normalization separately for each policy.

In [ ]:
calibration_config = ConfigTrainClassifier(
    dataset_name=DATASET_NAME, policy=replace(policy_spec, lora_config=None),
    reward=proxy_spec, judge=judge_spec,
    start_dataset=CALIBRATION_RANGE[0], end_dataset=CALIBRATION_RANGE[1],
    generation_batch_size=64, reward_batch_size=128, judge_batch_size=32,
    score_max_length=1024, gap_finder_batch_size=16,
)
if CALIBRATION_PATH.is_file():
    gap_calibration = functions.GapCalibration.load(CALIBRATION_PATH)
    print(f"Loaded {CALIBRATION_PATH}")
else:
    gap_calibration = functions.calculate_gap_calibration(calibration_config)
    gap_calibration.save(CALIBRATION_PATH)
gap_calibration

## 3. Stage-1 PPO with normalized proxy reward

This policy is the common parent of the later control and treatment, and supplies on-policy responses for training the two-head model.

In [ ]:
normalized_proxy_spec = replace(
    proxy_spec, mean=gap_calibration.proxy_mean, std=gap_calibration.proxy_std
)
stage1_config = TrainingPPOConfig(
    policy=policy_spec, reward=normalized_proxy_spec,
    dataset=DatasetSpec(dataset_name=DATASET_NAME, start=STAGE1_PPO_RANGE[0], end=STAGE1_PPO_RANGE[1]),
    output_dir=STAGE1_PATH.parent, epochs=1.0, batch_size=8,
    gradient_accumulation_steps=4, rollout_forward_batch_size=16,
    generation_batch_size=32, response_length=128,
    num_ppo_epochs=4, num_mini_batches=1, learning_rate=3e-6,
    save_steps=100, logging_steps=5,
)
set_seed(RANDOM_STATE)
if policy_checkpoint_ready(STAGE1_PATH):
    stage1_policy = PolicyModel.load(STAGE1_PATH, is_trainable=False)
    print(f"Loaded {STAGE1_PATH}")
else:
    stage1_policy = functions.temp_ppo_train_policy(stage1_config)
STAGE1_PATH

## 4. Collect 20k stage-1 responses and create a fixed split

Targets are the frozen normalized gap `proxy_z - judge_z`. The PPO, gap-training, and final-evaluation prompt ranges do not overlap.

In [ ]:
gap_data_config = replace(
    calibration_config,
    policy=replace(policy_spec, lora_config=None),
    start_dataset=GAP_DATA_RANGE[0], end_dataset=GAP_DATA_RANGE[1],
)
if GAP_DATA_PATH.is_file():
    gap_dataset = DatasetGapFinder.load(GAP_DATA_PATH)
    print(f"Loaded {GAP_DATA_PATH}")
else:
    gap_dataset = functions.collect_gap_finder_dataset(
        gap_data_config, gap_calibration, policy=stage1_policy
    )
    gap_dataset.save(GAP_DATA_PATH)
if len(gap_dataset) != 20_000:
    raise ValueError(f"Expected 20,000 gap examples, found {len(gap_dataset):,}")
release(stage1_policy)
train_gap, validation_gap, test_gap = gap_dataset.split_three_way(
    train_size=0.70, validation_size=0.15, test_size=0.15, random_state=RANDOM_STATE
)
assert (len(train_gap), len(validation_gap), len(test_gap)) == (14_000, 3_000, 3_000)
pd.Series({
    "train": len(train_gap), "validation": len(validation_gap), "test": len(test_gap),
    "train_high_gap_fraction": np.mean([r["labels"] > gap_calibration.theta for r in train_gap.dataset]),
})

## 5. Train/load the selected two-head setup and select `tau` on validation

The architecture and loss weights are fixed from the preceding ablation: `alpha=3`, `lambda=1`. Only the operating threshold is selected here, using validation labels only.

In [ ]:
set_seed(RANDOM_STATE)
if (TWO_HEAD_PATH / TwoHeadGapFinder.METADATA_FILE_NAME).is_file():
    two_head = TwoHeadGapFinder.load(TWO_HEAD_PATH)
    print(f"Loaded {TWO_HEAD_PATH}")
else:
    two_head = TwoHeadGapFinder(
        MODEL_NAME, theta=gap_calibration.theta, gap_finder_id=gap_dataset.id,
        source_policy=str(STAGE1_PATH),
    )
    two_head_training = TwoHeadGapFinderTrainingConfig(
        output_dir=str(TWO_HEAD_RUN), theta=gap_calibration.theta, epochs=3.0,
        batch_size=16, gradient_accumulation_steps=8, gradient_checkpointing=True,
        learning_rate=2e-5, max_length=512,
        high_gap_weight=HIGH_GAP_WEIGHT, detector_loss_weight=DETECTOR_LOSS_WEIGHT,
        lora_settings=LoRASettings(),
    )
    TwoHeadGapFinderTrainer(two_head, two_head_training).train(train_gap, validation_gap)

def gap_arrays(dataset):
    prompts = [row["prompt"] for row in dataset.dataset]
    answers = [row["answer"] for row in dataset.dataset]
    gaps = np.asarray([row["labels"] for row in dataset.dataset], dtype=np.float64)
    return prompts, answers, gaps

validation_prompts, validation_answers, actual_validation_gaps = gap_arrays(validation_gap)
predicted_validation_gaps, validation_probabilities = two_head.predict(
    validation_prompts, validation_answers, batch_size=16, max_length=512
)
predicted_validation_gaps = np.asarray(predicted_validation_gaps)
validation_probabilities = np.asarray(validation_probabilities)
actual_validation_tail = actual_validation_gaps > gap_calibration.theta
threshold_rows = []
for threshold in THRESHOLDS:
    predicted_tail = validation_probabilities >= threshold
    threshold_rows.append({
        "threshold": float(threshold),
        "precision": float(precision_score(actual_validation_tail, predicted_tail, zero_division=0)),
        "recall": float(recall_score(actual_validation_tail, predicted_tail, zero_division=0)),
        "f1": float(f1_score(actual_validation_tail, predicted_tail, zero_division=0)),
    })
threshold_table = pd.DataFrame(threshold_rows)
threshold_table.to_csv(TWO_HEAD_RUN / "validation_threshold_curve.csv", index=False)
eligible = threshold_table[threshold_table["recall"] >= MIN_VALIDATION_RECALL]
if eligible.empty:
    raise RuntimeError("No detector threshold reaches the required validation recall")
selected_threshold_row = eligible.sort_values(
    ["f1", "precision", "threshold"], ascending=[False, False, False]
).iloc[0]
DETECTOR_THRESHOLD = float(selected_threshold_row["threshold"])
CORRECTION_MAX_GAP = float(np.quantile(
    np.maximum([row["labels"] for row in train_gap.dataset], 0.0),
    CORRECTION_CAP_QUANTILE,
))
threshold_artifact = {
    "detector_threshold": DETECTOR_THRESHOLD,
    "minimum_validation_recall": MIN_VALIDATION_RECALL,
    "correction_max_gap": CORRECTION_MAX_GAP,
    **selected_threshold_row.to_dict(),
}
(TWO_HEAD_RUN / "correction_config.json").write_text(
    json.dumps(threshold_artifact, indent=2, sort_keys=True) + "\n"
)
threshold_artifact

## 6. Verify the correction mechanism before spending more PPO compute

This untouched gap-model test asks whether corrected proxy scores are closer to judge scores than raw proxy scores. It also reports detector generalization and the effect of the safety cap.

In [ ]:
test_prompts, test_answers, actual_test_gaps = gap_arrays(test_gap)
predicted_test_gaps, test_probabilities = two_head.predict(
    test_prompts, test_answers, batch_size=16, max_length=512
)
predicted_test_gaps = np.asarray(predicted_test_gaps)
test_probabilities = np.asarray(test_probabilities)
gap_test_report = compute_two_head_report(
    actual_test_gaps, predicted_test_gaps, test_probabilities,
    theta=gap_calibration.theta, detector_threshold=DETECTOR_THRESHOLD,
)
test_proxy_z = np.asarray([row["proxy_score"] for row in test_gap.dataset])
test_judge_z = np.asarray([row["judge_score"] for row in test_gap.dataset])
test_intervention = test_probabilities >= DETECTOR_THRESHOLD
bounded_test_gap = np.clip(predicted_test_gaps, 0.0, CORRECTION_MAX_GAP)
test_corrected_proxy_z = test_proxy_z - test_intervention * bounded_test_gap
gap_test_report.update({
    "raw_proxy_vs_judge_mse": float(mean_squared_error(test_judge_z, test_proxy_z)),
    "corrected_proxy_vs_judge_mse": float(mean_squared_error(test_judge_z, test_corrected_proxy_z)),
    "raw_proxy_vs_judge_mae": float(mean_absolute_error(test_judge_z, test_proxy_z)),
    "corrected_proxy_vs_judge_mae": float(mean_absolute_error(test_judge_z, test_corrected_proxy_z)),
    "intervention_fraction": float(test_intervention.mean()),
})
gap_test_report["mse_improvement_fraction"] = 1.0 - (
    gap_test_report["corrected_proxy_vs_judge_mse"] / gap_test_report["raw_proxy_vs_judge_mse"]
)
gap_test_report["mae_improvement_fraction"] = 1.0 - (
    gap_test_report["corrected_proxy_vs_judge_mae"] / gap_test_report["raw_proxy_vs_judge_mae"]
)
(TWO_HEAD_RUN / "gap_test_report.json").write_text(
    json.dumps(gap_test_report, indent=2, sort_keys=True) + "\n"
)
pd.Series(gap_test_report)

## 7. Matched stage-2 PPO: proxy control versus corrected treatment

Both branches warm-start from the exact stage-1 checkpoint, use the exact same prompts and hyperparameters, and receive the same random seed. Their only intended difference is the reward adjustment.

In [ ]:
# The two-head model is not needed during the control run; keep it on CPU.
two_head.offload()
functions.empty_cuda_cache()
stage2_policy_spec = replace(policy_spec, checkpoint=STAGE1_PATH)
stage2_base_config = TrainingPPOConfig(
    policy=stage2_policy_spec, reward=normalized_proxy_spec,
    dataset=DatasetSpec(dataset_name=DATASET_NAME, start=STAGE2_PPO_RANGE[0], end=STAGE2_PPO_RANGE[1]),
    epochs=1.0, batch_size=8, gradient_accumulation_steps=4,
    rollout_forward_batch_size=16, generation_batch_size=32, response_length=128,
    num_ppo_epochs=4, num_mini_batches=1, learning_rate=3e-6,
    save_steps=100, logging_steps=5,
)

set_seed(RANDOM_STATE)
if policy_checkpoint_ready(CONTROL_PATH):
    control_policy = PolicyModel.load(CONTROL_PATH, is_trainable=False)
    print(f"Loaded {CONTROL_PATH}")
else:
    control_policy = functions.temp_ppo_train_policy(
        replace(stage2_base_config, output_dir=CONTROL_PATH.parent)
    )
release(control_policy)

correction = TwoHeadGapFinderCorrection(
    two_head, reward_std=gap_calibration.proxy_std,
    detector_threshold=DETECTOR_THRESHOLD, max_gap=CORRECTION_MAX_GAP,
)
set_seed(RANDOM_STATE)
if policy_checkpoint_ready(TREATMENT_PATH):
    treatment_policy = PolicyModel.load(TREATMENT_PATH, is_trainable=False)
    print(f"Loaded {TREATMENT_PATH}")
else:
    treatment_policy = functions.temp_ppo_train_policy(
        replace(stage2_base_config, output_dir=TREATMENT_PATH.parent),
        adjustments=(correction,),
    )
release(treatment_policy, two_head)
{"control": str(CONTROL_PATH), "treatment": str(TREATMENT_PATH)}

## 8. Generate responses on identical held-out prompts

Stage 1 is included as a reference. Control vs treatment is the causal comparison. Generation uses the same seed for each policy.

In [ ]:
raw_dataset = functions.load_dataset(DATASET_NAME)
evaluation_requests = RequestDataset.from_raw(
    raw_dataset, MODEL_NAME, start=FINAL_EVAL_RANGE[0], end=FINAL_EVAL_RANGE[1]
)
evaluation_prompts = list(evaluation_requests["prompts"])
policy_paths = {"stage1": STAGE1_PATH, "control": CONTROL_PATH, "treatment": TREATMENT_PATH}
answers_by_policy = {}
for policy_name, checkpoint in policy_paths.items():
    set_seed(EVALUATION_SEED)
    policy = PolicyModel.load(checkpoint, is_trainable=False)
    policy.generate_new_dataset(evaluation_requests, batch_size=32)
    answers_by_policy[policy_name] = list(policy.get_dataset_col("answers"))
    # Save the held-out dataset beside the checkpoint for optional Prometheus evaluation.
    policy.save_dataset(checkpoint)
    release(policy)
assert all(len(answers) == len(evaluation_prompts) for answers in answers_by_policy.values())
len(evaluation_prompts)

## 9. Score every response with proxy, judge, and the frozen two-head model

Models are loaded sequentially to control VRAM. The reference calibration remains fixed across all policies.

In [ ]:
policy_order = list(policy_paths)
all_prompts = [prompt for name in policy_order for prompt in evaluation_prompts]
all_answers = [answer for name in policy_order for answer in answers_by_policy[name]]
policy_labels = [name for name in policy_order for _ in evaluation_prompts]

proxy_model = RewardModel(proxy_spec.model_name, "proxy")
raw_proxy_scores = np.asarray(proxy_model.score(
    all_prompts, all_answers, batch_size=128, max_length=1024
))
release(proxy_model)
judge_model = RewardModel(judge_spec.model_name, "judge")
raw_judge_scores = np.asarray(judge_model.score(
    all_prompts, all_answers, batch_size=32, max_length=1024
))
release(judge_model)
two_head = TwoHeadGapFinder.load(TWO_HEAD_PATH)
predicted_gaps, detector_probabilities = two_head.predict(
    all_prompts, all_answers, batch_size=16, max_length=512
)
release(two_head)
predicted_gaps = np.asarray(predicted_gaps)
detector_probabilities = np.asarray(detector_probabilities)

proxy_z = (raw_proxy_scores - gap_calibration.proxy_mean) / gap_calibration.proxy_std
judge_z = (raw_judge_scores - gap_calibration.judge_mean) / gap_calibration.judge_std
actual_gap = proxy_z - judge_z
intervention = detector_probabilities >= DETECTOR_THRESHOLD
bounded_predicted_gap = np.clip(predicted_gaps, 0.0, CORRECTION_MAX_GAP)
corrected_proxy_z = proxy_z - intervention * bounded_predicted_gap

evaluation_rows = pd.DataFrame({
    "policy": policy_labels, "prompt": all_prompts, "answer": all_answers,
    "proxy_raw": raw_proxy_scores, "judge_raw": raw_judge_scores,
    "proxy_z": proxy_z, "judge_z": judge_z, "actual_gap": actual_gap,
    "predicted_gap": predicted_gaps, "detector_probability": detector_probabilities,
    "intervention": intervention, "corrected_proxy_z": corrected_proxy_z,
})
evaluation_rows.to_csv(OUTPUT_ROOT / "held_out_response_scores.csv", index=False)
evaluation_rows.head()

## 10. Policy-level outcome and reward-fidelity metrics

Primary policy outcome: mean frozen-normalized judge score. Reward-hacking indicators: proxy–judge gap, high-gap rate, and proxy/judge correlation. Mechanism metrics show whether the corrected reward is closer to the judge.

In [ ]:
def safe_auc(labels, probabilities, metric):
    return float(metric(labels, probabilities)) if np.unique(labels).size == 2 else float("nan")

def summarize_policy(frame):
    actual_tail = frame["actual_gap"].to_numpy() > gap_calibration.theta
    predicted_tail = frame["intervention"].to_numpy(dtype=bool)
    probabilities = frame["detector_probability"].to_numpy()
    return {
        "examples": len(frame),
        "mean_proxy_z": frame["proxy_z"].mean(),
        "mean_judge_z": frame["judge_z"].mean(),
        "mean_proxy_minus_judge_gap": frame["actual_gap"].mean(),
        "mean_absolute_gap": frame["actual_gap"].abs().mean(),
        "proxy_judge_mse": mean_squared_error(frame["judge_z"], frame["proxy_z"]),
        "corrected_judge_mse": mean_squared_error(frame["judge_z"], frame["corrected_proxy_z"]),
        "proxy_judge_mae": mean_absolute_error(frame["judge_z"], frame["proxy_z"]),
        "corrected_judge_mae": mean_absolute_error(frame["judge_z"], frame["corrected_proxy_z"]),
        "proxy_judge_pearson": frame[["proxy_z", "judge_z"]].corr(method="pearson").iloc[0, 1],
        "proxy_judge_spearman": frame[["proxy_z", "judge_z"]].corr(method="spearman").iloc[0, 1],
        "high_gap_rate": actual_tail.mean(),
        "intervention_rate": predicted_tail.mean(),
        "detector_precision": precision_score(actual_tail, predicted_tail, zero_division=0),
        "detector_recall": recall_score(actual_tail, predicted_tail, zero_division=0),
        "detector_f1": f1_score(actual_tail, predicted_tail, zero_division=0),
        "detector_pr_auc": safe_auc(actual_tail, probabilities, average_precision_score),
        "detector_roc_auc": safe_auc(actual_tail, probabilities, roc_auc_score),
        "mean_answer_characters": frame["answer"].str.len().mean(),
    }

policy_metrics = pd.DataFrame({
    name: summarize_policy(evaluation_rows[evaluation_rows["policy"] == name])
    for name in policy_order
}).T
policy_metrics["reward_mse_improvement_fraction"] = 1.0 - (
    policy_metrics["corrected_judge_mse"] / policy_metrics["proxy_judge_mse"]
)
policy_metrics.to_csv(OUTPUT_ROOT / "policy_metrics.csv")
policy_metrics.T

## 11. Paired control-versus-treatment inference

Because both policies answer the same prompts, differences are paired by prompt. Bootstrap 95% intervals quantify uncertainty. A convincing result has a positive judge-score delta and reduced gap indicators; proxy score may decrease, which is acceptable.

In [ ]:
control = evaluation_rows[evaluation_rows["policy"] == "control"].reset_index(drop=True)
treatment = evaluation_rows[evaluation_rows["policy"] == "treatment"].reset_index(drop=True)
assert control["prompt"].equals(treatment["prompt"])

def paired_bootstrap(values, samples=10_000, seed=RANDOM_STATE):
    values = np.asarray(values, dtype=np.float64)
    rng = np.random.default_rng(seed)
    means = np.empty(samples)
    for index in range(samples):
        means[index] = values[rng.integers(0, len(values), len(values))].mean()
    low, high = np.quantile(means, [0.025, 0.975])
    return float(values.mean()), float(low), float(high)

comparisons = {
    "judge_z_delta_treatment_minus_control": treatment["judge_z"] - control["judge_z"],
    "proxy_z_delta_treatment_minus_control": treatment["proxy_z"] - control["proxy_z"],
    "absolute_gap_delta_treatment_minus_control": treatment["actual_gap"].abs() - control["actual_gap"].abs(),
    "high_gap_delta_treatment_minus_control": (
        (treatment["actual_gap"] > gap_calibration.theta).astype(float)
        - (control["actual_gap"] > gap_calibration.theta).astype(float)
    ),
}
paired_results = pd.DataFrame([
    {"metric": name, "mean_delta": result[0], "ci95_low": result[1], "ci95_high": result[2]}
    for name, values in comparisons.items()
    for result in [paired_bootstrap(values)]
]).set_index("metric")
paired_results.loc["judge_pairwise_win_rate", "mean_delta"] = float(
    (treatment["judge_z"] > control["judge_z"]).mean()
)
paired_results.to_csv(OUTPUT_ROOT / "paired_control_treatment.csv")
paired_results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
policy_metrics["mean_judge_z"].plot.bar(ax=axes[0], title="Held-out judge score")
policy_metrics[["mean_absolute_gap", "high_gap_rate"]].plot.bar(ax=axes[1], title="Reward-hacking indicators")
policy_metrics[["proxy_judge_mse", "corrected_judge_mse"]].plot.bar(ax=axes[2], title="Reward fidelity to judge")
for axis in axes:
    axis.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "policy_comparison.png", dpi=160, bbox_inches="tight")

## 12. Optional independent response-quality evaluation

The Skywork judge is the primary evaluator because it defines the gap. Prometheus provides an additional, independent quality check. It is expensive and is run sequentially.

In [ ]:
prometheus_scores = {}
if RUN_PROMETHEUS_EVALUATION:
    for policy_name, checkpoint in policy_paths.items():
        config = EvaluateConfig(
            policy=replace(policy_spec, checkpoint=checkpoint),
            evaluator=EvaluatorSpec(
                class_name="PrometheusEvaluator",
                model_name="prometheus-eval/prometheus-7b-v2.0",
            ),
            evaluator_batch_size=1, evaluator_reset=True,
        )
        prometheus_scores[policy_name] = functions.evaluate_policy(config)
    pd.Series(prometheus_scores, name="prometheus_score").to_csv(
        OUTPUT_ROOT / "prometheus_scores.csv"
    )
prometheus_scores

## 13. Inspect the largest treatment changes and write the final report

Do not rely only on aggregate scores. Inspect prompts where the treatment most improves or harms judge score, especially interventions and remaining high-gap failures.

In [ ]:
qualitative = pd.DataFrame({
    "prompt": control["prompt"],
    "control_answer": control["answer"],
    "treatment_answer": treatment["answer"],
    "control_judge_z": control["judge_z"],
    "treatment_judge_z": treatment["judge_z"],
    "judge_delta": treatment["judge_z"] - control["judge_z"],
    "control_gap": control["actual_gap"],
    "treatment_gap": treatment["actual_gap"],
    "treatment_detector_probability": treatment["detector_probability"],
    "treatment_intervention": treatment["intervention"],
})
qualitative.to_csv(OUTPUT_ROOT / "qualitative_pairs.csv", index=False)
display(qualitative.nlargest(10, "judge_delta"))
display(qualitative.nsmallest(10, "judge_delta"))

In [ ]:
judge_delta = paired_results.loc["judge_z_delta_treatment_minus_control"]
gap_delta = paired_results.loc["absolute_gap_delta_treatment_minus_control"]
tail_delta = paired_results.loc["high_gap_delta_treatment_minus_control"]
conclusions = {
    "judge_improved_point_estimate": bool(judge_delta["mean_delta"] > 0),
    "judge_improved_ci_excludes_zero": bool(judge_delta["ci95_low"] > 0),
    "absolute_gap_reduced_point_estimate": bool(gap_delta["mean_delta"] < 0),
    "absolute_gap_reduced_ci_excludes_zero": bool(gap_delta["ci95_high"] < 0),
    "high_gap_rate_reduced_point_estimate": bool(tail_delta["mean_delta"] < 0),
    "treatment_corrected_reward_closer_than_proxy": bool(
        policy_metrics.loc["treatment", "corrected_judge_mse"]
        < policy_metrics.loc["treatment", "proxy_judge_mse"]
    ),
    "detector_threshold": DETECTOR_THRESHOLD,
    "correction_max_gap": CORRECTION_MAX_GAP,
    "prometheus_scores": prometheus_scores,
}
(OUTPUT_ROOT / "experiment_conclusions.json").write_text(
    json.dumps(conclusions, indent=2, sort_keys=True) + "\n"
)
conclusions

## 14. Download the important experiment outcomes

This creates a compact ZIP with all reports, tables, plots, scored held-out responses, the frozen configuration, and the code needed to interpret the run. Multi-gigabyte checkpoints and the 20k training dataset are excluded by default; change the two flags below if you explicitly need them in the archive.

In [ ]:
from IPython.display import FileLink, display

INCLUDE_CHECKPOINTS_IN_DOWNLOAD = False
INCLUDE_20K_DATASET_IN_DOWNLOAD = False

manifest = {
    "dataset_name": DATASET_NAME,
    "model_name": MODEL_NAME,
    "random_state": RANDOM_STATE,
    "evaluation_seed": EVALUATION_SEED,
    "ranges": {
        "calibration": CALIBRATION_RANGE, "stage1_ppo": STAGE1_PPO_RANGE,
        "gap_data": GAP_DATA_RANGE, "stage2_ppo": STAGE2_PPO_RANGE,
        "final_evaluation": FINAL_EVAL_RANGE,
    },
    "two_head": {
        "high_gap_weight": HIGH_GAP_WEIGHT,
        "detector_loss_weight": DETECTOR_LOSS_WEIGHT,
        "detector_threshold": DETECTOR_THRESHOLD,
        "minimum_validation_recall": MIN_VALIDATION_RECALL,
        "correction_cap_quantile": CORRECTION_CAP_QUANTILE,
        "correction_max_gap": CORRECTION_MAX_GAP,
    },
    "ppo": {
        "epochs": 1.0, "batch_size": 8,
        "gradient_accumulation_steps": 4,
        "rollout_forward_batch_size": 16, "response_length": 128,
        "num_ppo_epochs": 4, "learning_rate": 3e-6,
    },
    "checkpoint_paths": {name: str(path) for name, path in policy_paths.items()},
}
manifest_path = OUTPUT_ROOT / "experiment_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")

result_artifacts = [
    CALIBRATION_PATH,
    TWO_HEAD_RUN / "correction_config.json",
    TWO_HEAD_RUN / "validation_threshold_curve.csv",
    TWO_HEAD_RUN / "gap_test_report.json",
    OUTPUT_ROOT / "held_out_response_scores.csv",
    OUTPUT_ROOT / "policy_metrics.csv",
    OUTPUT_ROOT / "paired_control_treatment.csv",
    OUTPUT_ROOT / "prometheus_scores.csv",
    OUTPUT_ROOT / "qualitative_pairs.csv",
    OUTPUT_ROOT / "policy_comparison.png",
    OUTPUT_ROOT / "experiment_conclusions.json",
    manifest_path,
]
code_artifacts = [
    Path("two_head_gap_finder_ppo_experiment.ipynb"),
    Path("Models/reward_adjustment.py"),
    Path("Models/model_two_head_gap_finder.py"),
    Path("Trainers/trainer_two_head_gap_finder.py"),
    Path("requirements.txt"),
]
if INCLUDE_20K_DATASET_IN_DOWNLOAD:
    result_artifacts.append(GAP_DATA_PATH)
if INCLUDE_CHECKPOINTS_IN_DOWNLOAD:
    for checkpoint in [STAGE1_PATH, TWO_HEAD_PATH, CONTROL_PATH, TREATMENT_PATH]:
        result_artifacts.extend(path for path in checkpoint.rglob("*") if path.is_file())

archive_path = OUTPUT_ROOT / "important_outcomes.zip"
included = []
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.writestr(
        "README.txt",
        "Two-head GapFinder PPO experiment outcomes.\n"
        "Model checkpoints and the 20k training dataset are excluded unless their export flags were enabled.\n",
    )
    for artifact in result_artifacts:
        artifact = Path(artifact)
        if artifact.is_file() and artifact.resolve() != archive_path.resolve():
            try:
                archive_name = Path("results") / artifact.relative_to(OUTPUT_ROOT)
            except ValueError:
                archive_name = Path("results") / artifact.name
            archive.write(artifact, archive_name)
            included.append(str(archive_name))
    for artifact in code_artifacts:
        if artifact.is_file():
            archive_name = Path("code") / artifact
            archive.write(artifact, archive_name)
            included.append(str(archive_name))

print(f"Created {archive_path.resolve()} ({archive_path.stat().st_size / 1_048_576:.2f} MiB)")
print(f"Included {len(included)} files")
display(FileLink(str(archive_path)))

## What counts as improvement?

The strongest evidence is: treatment judge score improves over the matched proxy-only control, its 95% paired interval excludes zero, and absolute/high-tail proxy–judge disagreement decreases without a collapse in independent response quality. A lower treatment proxy score together with a higher judge score is a success, not a failure.

A single seed is still exploratory evidence. For a paper-level claim, repeat the matched stage-2 control/treatment experiment with at least 3–5 seeds and report the distribution across seeds.